<a href="https://colab.research.google.com/github/abhimanyudalal1/FINE_TUNING_LLMS/blob/main/Copy_of_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error
from scipy.sparse import hstack
import lightgbm as lgb

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Load train and test data:

In [ ]:
DATASET_FOLDER = '/content/'
TEST_DATASET_FOLDER = '/content/'

train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test = pd.read_csv(os.path.join(TEST_DATASET_FOLDER, 'test.csv'))

Utility Functions:

In [ ]:
unit_map = {
    "ml": "ml", "milliliter": "ml", "milliliters": "ml", "millilitres": "ml", "millilitres": "ml",
    "l": "l", "liter": "l", "litre": "l",
    "oz": "oz", "ounce": "oz", "ounces": "oz",
    "fl oz": "fl_oz", "fluid ounce": "fl_oz", "fluid ounces": "fl_oz",
    "g": "g", "gram": "g", "grams": "g", "gm": "g",
    "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo gram": "kg", "kilo grams": "kg",
    "count": "count", "ct": "count", "pcs": "count", "piece": "count", "pack": "pack"
}

keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant"]

def clean_text(text):
    """Basic text cleaning for TF-IDF"""
    text = re.sub(r"â€“|â€|Ã|™", "", str(text))
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s\.\%\-]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_value(text):
    match = re.search(r"Value:\s*([\d\.]+)", str(text))
    return float(match.group(1)) if match else np.nan

def extract_unit(text):
    text = str(text).lower()
    for pattern, unit in unit_map.items():
        if re.search(rf'\b{pattern}\b', text):
            return unit
    return "unknown"

def extract_item_name(text):
    match = re.search(r"Item Name:\s*(.*?)\n", str(text))
    return match.group(1).strip().lower() if match else ""

def feature_engineering(df):
    """Feature engineering from catalog content"""
    df["catalog_content"] = df["catalog_content"].fillna("").apply(clean_text)
    df["Value"] = df["catalog_content"].apply(extract_value)
    df["Unit"] = df["catalog_content"].apply(extract_unit)
    df["Item_Name"] = df["catalog_content"].apply(extract_item_name)

    # Fill missing numeric features
    df["Value"] = df["Value"].fillna(df["Value"].median())
    df["Unit"] = df["Unit"].fillna("unknown")



    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").fillna(0)
    df["Value"] = df["Value"].clip(lower=0)
    df["log_value"] = np.log1p(df["Value"])
    df["word_count"] = df["catalog_content"].apply(lambda x: len(x.split()))
    df["char_count"] = df["catalog_content"].apply(len)
    df["digit_ratio"] = df["catalog_content"].apply(lambda x: sum(c.isdigit() for c in x) / max(len(x), 1))
    df["avg_word_len"] = df["catalog_content"].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) else 0)

    df["bullet_count"] = df["catalog_content"].str.count("bullet point")


    # Keyword flags (you can tune/add more keywords as you analyze data)
    for kw in keywords:
        df[f"has_{kw}"] = df["catalog_content"].str.contains(kw, case=False).astype(int)

    return df

Feature Engineering:

In [ ]:
train.drop('sample_id', axis=1, inplace=True)
train.drop('image_link', axis=1, inplace=True)

train = feature_engineering(train)
test = feature_engineering(test)

# Encode Unit
le = LabelEncoder()
train["Unit_enc"] = le.fit_transform(train["Unit"])
test["Unit_enc"] = le.transform(test["Unit"].map(lambda x: x if x in le.classes_ else "unknown"))

TF-IDF Features:

In [ ]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    stop_words=None,
    analyzer='word'
)
# tfidf_name = TfidfVectorizer(
#     max_features=5000,
#     ngram_range=(1, 2),
#     min_df=1,
#     max_df=0.95,
#     sublinear_tf=True,
#     stop_words=None,
#     analyzer='word'
# )

# Clean text just to be safe
# train["Item_Name"] = train["Item_Name"].fillna("").astype(str)
# test["Item_Name"] = test["Item_Name"].fillna("").astype(str)

train["catalog_content"] = train["catalog_content"].fillna("").astype(str)
test["catalog_content"] = test["catalog_content"].fillna("").astype(str)

tfidf_train = tfidf.fit_transform(train["catalog_content"])
tfidf_test = tfidf.transform(test["catalog_content"])
# Item_train = tfidf_name.fit_transform(train["Item_Name"])
# Item_test = tfidf_name.transform(test["Item_Name"])

Combine Structured Features:

In [ ]:
structured_features = ["Value", "Unit_enc", "log_value", "word_count", "char_count",
    "digit_ratio", "avg_word_len", "bullet_count"] + [f"has_{kw}" for kw in keywords]
scaler = StandardScaler()
train_struct = scaler.fit_transform(train[structured_features])
test_struct = scaler.transform(test[structured_features])

X = hstack([tfidf_train, train_struct])
X = X.tocsr()
X_test = hstack([tfidf_test, test_struct])
X_test = X_test.tocsr()
y = np.log1p(train["price"].values)

In [ ]:
train.head(10)
# train.columns

LightGBM Training:

In [ ]:
# Define SMAPE function
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred) + 0.000001) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0  # avoid division by zero
    return 100 * np.mean(diff)

# K-Fold parameters
params = {
    "objective": "regression",
    "metric": "mae",  # still using MAE internally
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 22,
    "device": "gpu",
}

kf = KFold(n_splits=5, shuffle=True, random_state=22)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
best_iterations = []

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n----- Fold {fold+1} -----")

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)],
    )
    best_iterations.append(model.best_iteration)

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

    # Compute SMAPE for this fold
    fold_smape = smape(y_val, oof_preds[val_idx])
    print(f"Fold {fold+1} SMAPE: {fold_smape:.4f}")

# Overall CV SMAPE
overall_smape = smape(y, oof_preds)
print("\nOverall CV SMAPE:", overall_smape)
average_best_iteration = int(np.mean(best_iterations))

Final Model Training:

In [ ]:
# Prepare full dataset
full_train_set = lgb.Dataset(X, label=y)

# Train final model on all data
final_model = lgb.train(
    params,
    full_train_set,
    num_boost_round=average_best_iteration
)

Final Model Prediction:

In [ ]:
final_preds_log = final_model.predict(X_test, num_iteration=final_model.best_iteration)
final_preds = np.expm1(final_preds_log)
print(final_preds)

Save Predictions:

In [ ]:
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "price": final_preds
})

submission.to_csv(r"/content/predictions_lgbm.csv", index=False)
print(submission.head())

In [ ]:
import joblib
joblib.dump(final_model, "/content/final_lgbm_model.pkl")
joblib.dump(tfidf, "/content/tfidf_vectorizer.pkl")
joblib.dump(le, "/content/label_encoder.pkl")